In [ ]:
from cme.diffusion_models import diffusion as dd
from cme.simulators import diffusion_random_walk as rw
import pandas as pd
import numpy as np
from cme.utils import common_utils as ut
import arviz as az
import os
import numpy as np

In [ ]:
import pymc as pm

In [ ]:
from IPython.display import clear_output, DisplayHandle
def update_patch(self, obj):
    clear_output(wait=True)
    self.display(obj)
DisplayHandle.update = update_patch

In [ ]:
rotation_RT = pd.read_csv(f"data/rotation_rt.csv")
#rotation_RT = pd.read_csv(f"data/sim_low_low_500_rt.csv")
rotation_RT_n = rotation_RT.loc[~rotation_RT.isna().any(axis=1),:].to_numpy()#[0:1,:]

rotation_X = pd.read_csv(f"data/rotation_ra.csv")
#rotation_X = pd.read_csv(f"data/sim_low_low_500_ra.csv")
rotation_X_n = rotation_X.loc[~rotation_RT.isna().any(axis=1),:].astype(int).to_numpy()#[0:1,:]

In [ ]:
I, J = rotation_RT_n.shape

In [ ]:
v_p = np.asarray(pm.Normal.dist(0,1,shape=(1,)).eval())
v_p

In [ ]:
v_p

In [ ]:
theta, alpha, tau, sigma = 100, 1.5, 0.01, 1
#v_p=np.asarray([-0.62683146])
RT_mat, X_mat,v_arr,tr_arr = rw.gen_RT_X_mat(theta, alpha, tau, sigma, v_p, I=2, J = 5, process="Wiener")



In [ ]:
len([[],[]])

In [ ]:
def diffusion_sim(rng, theta, alpha, tau, sigma, v_p, size):
    if size is None:
        I,J = 1,1
    elif len(size) > 1:
        I, J = size
    else:
        I, J = size,1

    
    RT_mat, X_mat,v_arr,tr_arr = rw.gen_RT_X_mat(theta, alpha, tau, sigma, v_p, I=I, J = J, process="Wiener")
    return RT_mat

In [ ]:
I,J=10,5

In [ ]:
with pm.Model() as model:
    v = pm.Normal("v",0,1,shape=(1,))
    #a = pm.Gamma("a",3,2)
    #z = pm.Beta("z", 1,1)
    #t_er = pm.Beta("ter", 1,1)
    RT_sim = pm.Simulator("RT", diffusion_sim, theta, alpha, tau, sigma, v, observed=rotation_RT_n)
    

    

In [ ]:
with model:
    prior_chain = pm.sample_prior_predictive(samples=10)

In [ ]:
prior_chain

In [ ]:
#import arviz as az
#az.plot_ppc(prior_chain, group="prior")

In [ ]:
with model:
    posterior_chain = pm.sample_smc(draws=200)